# Notebook 00: Dataset Profiling
## Amazon Fine Food Reviews — Raw Data Profile

This notebook profiles the raw dataset in the `/raw` zone to produce a JSON 
summary for data quality decisions in the report. 
The profiling logic is defined in `modules/profile_data.py`.

### Step 1: Configure Storage Access
Retrieves the storage account key from Azure Key Vault via Databricks secret scope.
No credentials have been hardcoded.

In [0]:
STORAGE_ACCOUNT = "stcw2bdcbh"
STORAGE_KEY = dbutils.secrets.get(scope="kv-scope", key="storage-account-key")
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

### Step 2: Define Profile Function
Computes row count, null counts, duplicate count, score distribution, 
time range, and candidate PII columns in a single Spark pass where possible.

In [0]:
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def profile_dataset(spark: SparkSession, input_path: str, file_format: str = "csv") -> dict:
    reader = spark.read
    if file_format == "csv":
        df = reader.option("header", True).option("inferSchema", True).csv(input_path)
    elif file_format == "json":
        df = reader.json(input_path)
    elif file_format == "parquet":
        df = reader.parquet(input_path)
    else:
        raise ValueError(f"Unsupported file_format: {file_format}")

    profile = {
        "input_path": input_path,
        "format": file_format,
        "profiled_at_utc": datetime.utcnow().isoformat(),
        "row_count": df.count(),
        "column_count": len(df.columns),
        "schema": [{"name": f.name, "type": str(f.dataType)} for f in df.schema.fields],
        "null_counts": {},
        "duplicate_id_count": None,
        "score_distribution": {},
        "time_range": {},
        "candidate_pii_columns": [],
    }

    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_row = df.agg(*null_exprs).collect()[0].asDict()
    profile["null_counts"] = {c: int(null_row[c]) for c in df.columns}

    if "Id" in df.columns:
        total = profile["row_count"]
        distinct_ids = df.select("Id").distinct().count()
        profile["duplicate_id_count"] = total - distinct_ids

    if "Score" in df.columns:
        score_rows = df.filter(F.col("Score").cast("int").isNotNull()) \
            .groupBy(F.col("Score").cast("int").alias("Score")) \
            .count().orderBy("Score").collect()
        profile["score_distribution"] = {
            int(r["Score"]): int(r["count"]) for r in score_rows
        }

    if "Time" in df.columns:
        df_clean = df.filter(F.col("Time").cast("long").isNotNull())
        bounds = df_clean.agg(
            F.min(F.col("Time").cast("long")).alias("min_t"),
            F.max(F.col("Time").cast("long")).alias("max_t")
        ).collect()[0]
        if bounds["min_t"] is not None:
            profile["time_range"] = {
                "min_epoch": int(bounds["min_t"]),
                "max_epoch": int(bounds["max_t"]),
                "min_date": datetime.utcfromtimestamp(int(bounds["min_t"])).isoformat(),
                "max_date": datetime.utcfromtimestamp(int(bounds["max_t"])).isoformat(),
            }

    pii_keywords = ("user", "name", "email", "address", "phone")
    profile["candidate_pii_columns"] = [
        c for c in df.columns if any(k in c.lower() for k in pii_keywords)
    ]

    return profile

### Step 3: Run Profiler Against Full Dataset
Profiles the full 568,454-row CSV from the raw zone. 
Output is saved to `docs/profile.json` in the repository.

In [0]:
profile = profile_dataset(
    spark,
    "abfss://raw@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/Reviews.csv",
    "csv"
)
print(json.dumps(profile, indent=2))

{
  "input_path": "abfss://raw@stcw2bdcbh.dfs.core.windows.net/amazon-reviews/Reviews.csv",
  "format": "csv",
  "profiled_at_utc": "2026-04-30T19:54:55.689068",
  "row_count": 568454,
  "column_count": 10,
  "schema": [
    {
      "name": "Id",
      "type": "IntegerType()"
    },
    {
      "name": "ProductId",
      "type": "StringType()"
    },
    {
      "name": "UserId",
      "type": "StringType()"
    },
    {
      "name": "ProfileName",
      "type": "StringType()"
    },
    {
      "name": "HelpfulnessNumerator",
      "type": "StringType()"
    },
    {
      "name": "HelpfulnessDenominator",
      "type": "StringType()"
    },
    {
      "name": "Score",
      "type": "StringType()"
    },
    {
      "name": "Time",
      "type": "StringType()"
    },
    {
      "name": "Summary",
      "type": "StringType()"
    },
    {
      "name": "Text",
      "type": "StringType()"
    }
  ],
  "null_counts": {
    "Id": 0,
    "ProductId": 0,
    "UserId": 0,
    "ProfileNam